## GPT prompting: n80 examples, second filter

### requires python >= 3.10

In [2]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [3]:
sys.path.append("../../../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [4]:
from prompts.semantic_categories.v02.prompt import SYSTEM_PROMPT, FEW_SHOTS_STR, FEW_SHOTS

In [5]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [6]:
RESULTS_DIR = "../../results/"

# algse 10K lause jaokos
#EXAMPLE_FILE = "n80_examples_large_v01/gpt_v01/" + "gpt_10K_b12_run01.csv"
#GPT_ANSWER_FILE = "n80_examples_large_v01/gpt_v02/"+ "gpt_10K_b10_run01.csv"

# esimesest filtrist tulnud laused
EXAMPLE_FILE = RESULTS_DIR + "n80_examples_large_v02/gpt_v01/" + "gpt_b10_run01.csv"
# teise filtri tulemusfail
GPT_ANSWER_FILE = RESULTS_DIR + "n80_examples_large_v02/gpt_v02/"+ "gpt_b10_run01.csv"

CONF_FILE = 'azure.ini'


# OSA I : Andmed


## testimise põhjusel on kasutusel vana 10k v1 andmefail, et tulemusi saaks võrrelda

In [7]:
df = pd.read_csv(EXAMPLE_FILE, encoding="utf-8",  sep=",")

In [8]:
len(df)

30

In [9]:
# shufflida, et ei oleks alati sama järjekord gpt-le andes
spatial_obl_ex = df.sample(frac=1)

In [10]:
spatial_obl_ex

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
16,2149636,3433963,8,ringlema,NaN,in,piletiäri,piletiäris,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The word 'piletiäris' refers to ticket business or trade, which is not a location but rather an activity or field of work."
5,8099731,12997190,8,pöörama,NaN,ill,langus,langusesse,"Pigem on see madal või pöörab sootuks langusesse . """,NaN,NaN,NaN,no,"The phrase 'langusesse' was classified as not a location because it refers to a state of decline, not a physical place."
7,12633679,20215816,2,toimuma,NaN,ill,krae,kraesse,Raudteelaste kraesse toimunud õnnetusi Rentiku sõnul siiski veeretada ei saa .,NaN,NaN,NaN,no,"The phrase 'kraesse' was classified as not a location because it is used metaphorically to mean attribution of blame, not a physical place."
17,12372585,19808269,6,sööma,NaN,in,fuajee,fuajees,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN
28,6046200,9704021,8,müüma,NaN,ill,lennuk,lennukitesse,"Turismifirmad on müünud kalli raha eest pileteid lennukitesse , mis tõuseksid õhku samaks ajaks kui maa suurim tehiskaaslane maale tagasi jõuab .",NaN,NaN,NaN,no,"The phrase 'lennukitesse' refers to entry into planes, which are not geographical locations, so it was classified as not location ('no')."
6,10060758,16137721,6,jõudma,tagasi,el,lisaraha,lisarahast,"200 miljonit ettevõtete kätte jäävast lisarahast jõuab riigile tagasi üksikisiku tulumaksuna , käibemaksuna ja aktsiisidena .",NaN,NaN,NaN,no,"The phrase 'lisarahast' was classified as not a location because it refers to additional resources or money, not a physical place."
15,2792027,4477403,1,sadama,maha,in,Tallinn,Tallinnas,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN
12,2066247,3296559,23,pagema,NaN,adit,peit,peitu,"Valdo Lips lõi tempo alla , Ivo Saksakulm ja Chris Moore kontrollisid lauavõitlust , Indrek Varblane ja Andrus Nagel pagesid kaitse haardest peitu ja tulistasid kaugvisetega .",NaN,NaN,NaN,no,"The word 'peitu' means to hide and describes a state, not a location."
20,17113251,26300294,16,laskma,NaN,ill,kaabel,kaablisse,Ekraan ( sukk ) oli enamasti « ülihõre » ja laskis väljaspoolt kõik = mürad kaablisse .,NaN,NaN,NaN,no,"The phrase 'kaablisse' refers to an object or medium, not a location, so it was classified as not location ('no')."
3,15500381,24205992,12,panema,NaN,adit,hooldeprojekt,hooldeprojekti,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .",NaN,NaN,NaN,no,"The phrase 'hooldeprojekti' was classified as not a location because it refers to a project or initiative, not a physical place."


# OSA II : GPT

## GPT jaoks vajalik

In [11]:
config = configparser.ConfigParser()

status = config.read(CONF_FILE) 
assert status == [CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [12]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

In [13]:
SYSTEM_PROMPT

'\nYou are a classification assistant.\nIn this task location refers to "adverbial of place" (Estonian: kohamäärus) or "locative adverb".\nYour task: Given a list of examples, each with keys "l" (sentence) and "c" (phrase), classify whether "c" functions as a location in the context of the sentence.\nAdverbial of place answers to the question “where” (kus?/kuhu?/kust?) in the context of the sentence.\nIt is a place or concept where something or someone is located, goes to or comes from.\nCriteria:\n- concrete place (bank, table, Berlin)\n- abstract (literature, soul, TV channels, government, top of a group, history, thought, domain)\n- inanimate (journal, chair, wifi, bag, medal, toy, food, computer, wire, body parts)\n- alive (mother, Peter, dog, doctor, teacher)\n- event (dress rehearsal, camp, class, situation, meeting)\n- state or condition conceptualized as space (life, trouble, consciousness, attitude)\n- Locations ARE NOT phrases that show time, state of being, owner, experience

In [6]:
#FEW_SHOTS_STR

## Andmete söötmine

In [14]:

def classify_batch(my_batch, few_shots, system_prompt, client, deployment):
    """Gets a yes/no answer for a batch of sentences and phrases. 
    """
    
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": few_shots,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":   json.dumps(user_payload, ensure_ascii=False) }
        ]

        #return None, None
        response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(my_batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(my_batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


In [15]:

def explain_non_locations(
    client, 
    deployment,
    batch: List[Dict[str, str]],
    yes_no_results: List[str],
    yes_subset_ratio: float = 0.0,

) -> Dict[int, str]:

    # Determine which indices to explain
    no_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "no"]
    yes_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "yes"]

    # Diagnostic subset
    diag_count = int(len(yes_indices) * yes_subset_ratio)
    diag_indices = yes_indices[:diag_count]

    explain_indices = no_indices + diag_indices
    if not explain_indices:
        return None, None, None

    items_to_explain = [
        {
            "index": i,
            "l": json.loads(batch[i])["l"],
            "c": json.loads(batch[i])["c"],
            "classification": yes_no_results[i]
        }
        for i in explain_indices
    ]

    messages = [
        {"role": "system", 
         "content": ("Explain why each phrase 'c' was classified as adverbial of place ('yes') or not adverbial of place ('no') in sentence 'l'." 
                       "Give one sentence answer."
                        "You MUST return only a pure JSON object, without markdown and code fences. "
                        "The output must be a mapping: {index: explanation}. "
                        "Do not include ```json or any backticks. Do not include commentary.")
        },
        {"role": "user", "content": (
            """For EACH item without missing any, return a JSON object mapping index → explanation in this format '{"0": "explanation", "3": "explanation"}'.\n"""
            "Items:\n" +  json.dumps(items_to_explain, ensure_ascii=False)
        )}
    ]
    #return None, None,None 
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
    )

    raw = response.choices[0].message.content.strip()

    # ---- Pydantic validation ----
    try:
        ClassificationAnswer(form = json.loads(raw))
    except ValidationError as e:
        raise ValueError(f"Invalid JSON structure returned in explanations:\n{e}")

    return response, raw, explain_indices


## NB! muuda max_allowed_tok kui vaja

In [16]:
df = spatial_obl_ex

In [18]:
results = []
results2 = []
responses = []
explanations = []
explanations_all = {}

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
# kui suure osa võtta "yes" vastustest "why" küsimusse
yes_subset_ratio = 0.2
batch_start_index = 0

# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 30000 #3200000


batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, FEW_SHOTS_STR, SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    # võtab välja kõik batchis olnud "no" ja mõne "yes" ja küsib why
    expl_response, batch_explanations, answered_idx = explain_non_locations(
            batch=batch,
            yes_no_results=result_yesno,
            yes_subset_ratio=yes_subset_ratio,
            client=client, 
            deployment=DEPLOYMENT
        )

    # mapping: vastused õige lause+fraasiga kokku
    if batch_explanations is not None:
        used_tokens += expl_response.usage.total_tokens
        
        explanations.append(json.loads(batch_explanations))
        
        # Map batch-local -> global indices
        for local_i, explanation in json.loads(batch_explanations).items():
            global_i = batch_start_index + int(local_i)
            explanations_all[global_i] = explanation

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    
    #break



3it [00:07,  2.38s/it]


In [19]:
used_tokens # 30 lauset, batch 10-> 11881 tokenit,

11881

In [20]:
len(results)

30

## andmed tabelisse 

### enne kontroll kas andmeid on puudu ja vastavad lüngad täita

In [21]:
faulty_batches = {}
faulty_answers = {}
num_full_batches = int(len(df)/bs)
partial_batches = False if num_full_batches*bs == len(df) else True

if len(results) == len(df):
    df["classification2"] = [r["a"] for r in results]

    new_explanations = []
    for i in range(len(df)):
        if i in explanations_all.keys():
            new_explanations.append( explanations_all[i])
        else:
            new_explanations.append("")
    
        
    df["explanation2"] = new_explanations 


else: # juhuks kui mudel ei anna õiget arvu vastuseid tagasi
    new_results = []
    new_explanations = []
    for b, (batchres, expl) in enumerate(zip(results2, explanations),start=0):
        expected_len = bs if b < num_full_batches else len(df)-(num_full_batches*bs)
        
        if len(batchres) != expected_len and b < num_full_batches: # pole poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(expected_len)]
            #print(num_full_batches, b, len(replacement))
            new_results += replacement
            new_explanations += replacement
            faulty_batches[b] = batchres
            faulty_answers[b] = expl
        elif len(batchres) != expected_len and b >= num_full_batches and partial_batches:  # on poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(len(expected_len))]
            new_results += replacement
            faulty_batches[b] = batchres
            new_explanations += replacement
            faulty_answers[b] = expl
        elif len(batchres) == expected_len: # kõik ok 
            new_results += [r["a"] for r in batchres]
            for i in range(len(batchres)):
                if str(i) in expl.keys():
                    new_explanations.append( expl[str(i)])
                else:
                    new_explanations.append("")
            
    
    df["classification2"] = new_results
    df["explanation2"] = new_explanations

    #df["explanation"] = new_explanations 

In [22]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
16,2149636,3433963,8,ringlema,NaN,in,piletiäri,piletiäris,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The word 'piletiäris' refers to ticket business or trade, which is not a location but rather an activity or field of work.",yes,The phrase 'piletiäris' was classified as 'yes' because it describes the location where the transactions or actions occur.
5,8099731,12997190,8,pöörama,NaN,ill,langus,langusesse,"Pigem on see madal või pöörab sootuks langusesse . """,NaN,NaN,NaN,no,"The phrase 'langusesse' was classified as not a location because it refers to a state of decline, not a physical place.",yes,
7,12633679,20215816,2,toimuma,NaN,ill,krae,kraesse,Raudteelaste kraesse toimunud õnnetusi Rentiku sõnul siiski veeretada ei saa .,NaN,NaN,NaN,no,"The phrase 'kraesse' was classified as not a location because it is used metaphorically to mean attribution of blame, not a physical place.",yes,
17,12372585,19808269,6,sööma,NaN,in,fuajee,fuajees,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN,yes,
28,6046200,9704021,8,müüma,NaN,ill,lennuk,lennukitesse,"Turismifirmad on müünud kalli raha eest pileteid lennukitesse , mis tõuseksid õhku samaks ajaks kui maa suurim tehiskaaslane maale tagasi jõuab .",NaN,NaN,NaN,no,"The phrase 'lennukitesse' refers to entry into planes, which are not geographical locations, so it was classified as not location ('no').",yes,
6,10060758,16137721,6,jõudma,tagasi,el,lisaraha,lisarahast,"200 miljonit ettevõtete kätte jäävast lisarahast jõuab riigile tagasi üksikisiku tulumaksuna , käibemaksuna ja aktsiisidena .",NaN,NaN,NaN,no,"The phrase 'lisarahast' was classified as not a location because it refers to additional resources or money, not a physical place.",no,The phrase 'lisarahast' was classified as 'no' because it refers to the source of extra money and not a location.
15,2792027,4477403,1,sadama,maha,in,Tallinn,Tallinnas,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN,yes,
12,2066247,3296559,23,pagema,NaN,adit,peit,peitu,"Valdo Lips lõi tempo alla , Ivo Saksakulm ja Chris Moore kontrollisid lauavõitlust , Indrek Varblane ja Andrus Nagel pagesid kaitse haardest peitu ja tulistasid kaugvisetega .",NaN,NaN,NaN,no,"The word 'peitu' means to hide and describes a state, not a location.",yes,
20,17113251,26300294,16,laskma,NaN,ill,kaabel,kaablisse,Ekraan ( sukk ) oli enamasti « ülihõre » ja laskis väljaspoolt kõik = mürad kaablisse .,NaN,NaN,NaN,no,"The phrase 'kaablisse' refers to an object or medium, not a location, so it was classified as not location ('no').",yes,
3,15500381,24205992,12,panema,NaN,adit,hooldeprojekt,hooldeprojekti,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .",NaN,NaN,NaN,no,"The phrase 'hooldeprojekti' was classified as not a location because it refers to a project or initiative, not a physical place.",no,The phrase 'hooldeprojekti' was classified as 'no' because it indicates a target or purpose rather than a place.


### salvestada tulemused faili

In [23]:
df.to_csv(GPT_ANSWER_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

## optional saving

fname1 = saving_fname[:-4] + "_faulty_batches.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(faulty_batches, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_faulty_answers.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(faulty_answers, f, ensure_ascii=False)

fname1 = saving_fname[:-4] + "_yesno.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(results2, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_why_reponses.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(explanations, f, ensure_ascii=False)

In [9]:
#df = pd.read_csv(GPT_ANSWER_FILE , encoding="utf-8",  sep=",")

In [24]:
print("yes:", len(df[df["classification2"]=="yes"])) 
print("yes explained:", len(df[(df["classification2"]=="yes") & (~df["explanation2"].isna())]))
print("no:", len(df[df["classification2"]=="no"])) 

yes: 18
yes explained: 18
no: 12
